# **Introduction to Embeddings with the OpenAI API**

This Jupyter Notebook documents my comprehensive learning notes and vector implementations for the **Introduction to Embeddings with the OpenAI API** course. It covers vector embeddings generation, dimensionality options, distance metrics, semantic search, classification, clustering, and RAG pipelines.

## **Chapter 1: Generating Embeddings and Dimensions**

We map text inputs to a dense multi-dimensional vector space representing semantic values.

### **1.1: OpenAI Models**
- `text-embedding-3-small`: 1536 dimensions (highly efficient, standard).
- `text-embedding-3-large`: 3072 dimensions (high accuracy, precise).

### **1.2: Embedding Dimension Pruning (The `dimensions` argument)**
The newer third-generation models support the `dimensions` API parameter, allowing developer to prune (shorten) vectors (e.g. from 1536 to 256) without losing substantial semantic correlation.

In [ ]:
# Embedding query execution template
# response = client.embeddings.create(
#     model='text-embedding-3-small',
#     input='Embedding vectors serve as quantitative feature coordinates.',
#     dimensions=256   # Prunes dimension size to 256
# )
print('Embedding generation template prepared.')

## **Chapter 2: Semantic Similarity Search & Evaluation**

To search texts matching a query, we measure the proximity between vectors using standard metrics.

### **2.1: Mathematical Formulas**

- **Cosine Similarity (Angular Proximity):**
  $$\text{sim}(u, v) = \frac{u \cdot v}{\|u\| \|v\|} = \frac{\sum_{i=1}^D u_i v_i}{\sqrt{\sum_{i=1}^D u_i^2} \sqrt{\sum_{i=1}^D v_i^2}}$$

- **Dot Product Similarity:**
  If vectors are normalized to unit length (L2 norm = 1), similarity simplifies to the dot product:
  $$\text{sim}(u, v) = u \cdot v = \sum_{i=1}^D u_i v_i$$

- **Euclidean (L2) Distance:**
  $$d(u, v) = \sqrt{\sum_{i=1}^D (u_i - v_i)^2}$$

- **Manhattan (L1) Distance:**
  $$d(u, v) = \sum_{i=1}^D |u_i - v_i|$$

In [ ]:
import numpy as np
import pandas as pd

# Cosine similarity function
def cosine_similarity(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

# Mock Database
corpus = pd.DataFrame({
    'doc': ['Feline sitting on rug.', 'Machine learning classification.', 'Sunny weather forecast.'],
    'vector': [
        [0.05, 0.12, 0.65],
        [0.82, 0.11, 0.04],
        [0.08, 0.79, 0.12]
    ]
})

# Mock Query Vector
query_vector = np.array([0.06, 0.15, 0.62])

# Compute semantic search similarities
corpus['sim'] = corpus['vector'].apply(lambda x: cosine_similarity(query_vector, x))
results = corpus.sort_values(by='sim', ascending=False)
print('Semantic Search Results:\n', results[['doc', 'sim']])

### **2.2: Visualization with t-SNE & PCA**
To visualize high-dimensional embeddings, we use dimensionality reduction techniques like PCA or t-SNE (t-Distributed Stochastic Neighbor Embedding) to map coordinates to a 2D plot.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Project 3D vectors into 2D using PCA
vectors = np.array(corpus['vector'].tolist())
pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

# Plot the coordinates
plt.figure(figsize=(5,3))
plt.scatter(coords[:, 0], coords[:, 1], color='purple')
for i, text in enumerate(corpus['doc']):
    plt.annotate(text[:12], (coords[i, 0], coords[i, 1]))
plt.title('PCA Projection of Embeddings')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.grid(True)
plt.show()

## **Chapter 3: Classification & Clustering on Embeddings**

Embeddings can represent features ($X$) for classical Scikit-Learn pipelines.

### **3.1: Supervised Sentiment Classification**
Using vector lists to train classifiers (e.g. Logistic Regression).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = np.array(corpus['vector'].tolist())
y = np.array([0, 1, 0])  # Mock labels (0: animal/nature, 1: technology)

clf = LogisticRegression()
clf.fit(X, y)
test_vector = np.array([[0.78, 0.15, 0.05]])
print('Predicted Class Label:', clf.predict(test_vector))

### **3.2: Unsupervised Topic Clustering**
Group unlabeled texts using K-Means clustering.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
kmeans.fit(X)
print('Cluster Labels:', kmeans.labels_)

## **Chapter 4: Retrieval-Augmented Generation (RAG)**

RAG queries a vector store to retrieve top-$k$ context, builds an augmented prompt, and feeds it into ChatGPT.

In [ ]:
# Complete RAG Pipeline Simulation
user_query = 'feline resting on carpet'
print(f'User Query: {user_query}')

# Step 1: Retrieve context
retrieved_context = results.iloc[0]['doc']
print(f'Retrieved Context: {retrieved_context}')

# Step 2: Augment system prompt
augmented_prompt = [
    {
        'role': 'system',
        'content': f'Answer the user query based ONLY on the context provided. Context: {retrieved_context}'
    },
    {
        'role': 'user',
        'content': user_query
    }
]
print('\nAugmented prompt structure sent to LLM:\n', augmented_prompt)